In [1]:
# TODAY WE HANDLE HUMAN-IN-THE-LOOP --- Pausing the graph for human approval

""" Today, my graph learns to stop and wait for a human. Right now every node
runs automatically from start to finish. Today, I will add an interrupt - mechanism that
pauses the graph at a specific node, prints the pending action to the console, and
waits for you to type APPROVE or REJECT  before continuing. This is critical
for any production RAG system that triggers expensive API calls, sends emails, or writes
to a database --- you never want those to fire without a human sign-off.

Deep Coding: Interrupt and Resume
interrupt_before: LangGraph's built-in mechanism for pausing before a node
executes. You pass a list of node to graph.compile(interrupt_before = [...]) 
and the graph automatically stops before those nodes and saves state to
the checkpoint. No extra code inside node itself
The Approval Node: Build a new human_approval_node that sits between review_node
and any expensive downstream action. It prints the pending answer, the reviewer's verdict,
and the token cost so far, then waits for console input.
Resume on Approve: After the human types APPROVE, the graph resumes from
the checkpoint and the interrupted node executes normally. After REJECT, the
graph routes to a rejected_node that logs the rejection and ends cleanly.
The Expensive Action Node: Build a mock send_report_node that simulates an expensive or 
irreversible action - in real use this would be sending an email,
writing to a database, or calling a paid third-party API. Today it 
just prints a confirmation, but the approval gate in front of it is real.
"""



" Today, my graph learns to stop and wait for a human. Right now every node\nruns automatically from start to finish. Today, I will add an interrupt - mechanism that\npauses the graph at a specific node, prints the pending action to the console, and\nwaits for you to type APPROVE or REJECT  before continuing. This is critical\nfor any production RAG system that triggers expensive API calls, sends emails, or writes\nto a database --- you never want those to fire without a human sign-off.\n\nDeep Coding: Interrupt and Resume\ninterrupt_before: LangGraph's built-in mechanism for pausing before a node\nexecutes. You pass a list of node to graph.compile(interrupt_before = [...]) \nand the graph automatically stops before those nodes and saves state to\nthe checkpoint. No extra code inside node itself\nThe Approval Node: Build a new human_approval_node that sits between review_node\nand any expensive downstream action. It prints the pending answer, the reviewer's verdict,\nand the token cost

In [2]:

# Importing necessary libraries


import os
import re
import json
from groq import Groq
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional, TypedDict, Literal
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv, find_dotenv
from dotenv import load_dotenv, find_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext
from langchain_community.tools.tavily_search import TavilySearchResults

Settings.embed_model = HuggingFaceEmbedding(
    model_name = 'BAAI/bge-m3'
)


load_dotenv(find_dotenv())

client = Groq()
webSearch = TavilySearchResults(max_results = 3)


# LOADING EXISTING VECTOR STORE

# first connecting to existing vector store
# connecting to existing vectorstore
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# creating the primary retrieval tool
'''apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)'''

print('🤖🛩️ Vector Store connection established ⚡')


# TOKEN USER TRACKER
@dataclass
class TokenUsage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_calls: int = 0

    def add(self, usage):
        self.prompt_tokens += usage.prompt_tokens
        self.completion_tokens += usage.completion_tokens
        self.total_calls += 1
    
    def cost_estimate(
            self,
            input_price_per_1m: float = 2.50,
            output_price_per_1m: float = 10.00
    ) -> float:
        input_cost = (self.prompt_tokens /1000000) * input_price_per_1m
        output_cost = (self.completion_tokens /1000000) * output_price_per_1m
        return round(input_cost + output_cost, 6)
    
    def report(self):
        print(f"\n📣 Token usage report")
        print(f"LLM calls : {self.total_calls}")
        print(f"Prompt tokens: {self.prompt_tokens}")
        print(f"Completion tokens: {self.completion_tokens}")
        print(f"Total tokens: {self.prompt_tokens + self.completion_tokens:,}")
        print(f"Est. cost (GPT-40 pricing): ${self.cost_estimate()}")


# global tracher that is set before each run
usage_tracker = TokenUsage()


# TOKEN aware LLM Caller

def tracked_llm_call(messages: list, system: str = "") -> str:
    """
    Wraps every Groq call so token usage is always captured.
    Drop-in replacement for direct client.chat.completions.create calls.
    
    """

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages= full_messages,
        temperature= 0,
    )

    usage_tracker.add(response.usage)
    return response.choices[0].message.content

# RETRIEVAL WITH CONFIDENCE SCORING
RELEVANCE_THRESHOLD = 0.5

def retrieve_with_confidence(query: str) -> tuple[list, float]:
    """Returns retrieved docs and the top chunk's confidence score.
    Uses cosine similarity to score from your vectore store.
    """

    # creating a native llamaindex retriever from my initialized index
    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)

    # reults is a list of (document, score) tuples
    # lower score = more similar in FAISS  (L2 distance); invert if needed
    # For Cosine similarity stores, higher = better

    if not results:
        return [], 0.0
    
    
    top_score = float(results[0].score) if results[0].score is not None else 0.0

    print(f"📣 Top retrieval score : {top_score:.3f} (threshold: {RELEVANCE_THRESHOLD})")
    return results, top_score

def format_chunks(docs: list) -> str:
    return "\n\n---\n\n".join([doc.node.get_content() for doc in docs])
        


# CRAG -> Retrieval quality gate

def corrective_retrieve(query: str) -> tuple[str, str]:
    """
    Returns (context_text, source) where source is 'local' or 'web'.
    Applies CRAG logic: low confidence -> discard local, use web fallback.
    """

    docs, top_score = retrieve_with_confidence(query)

    if top_score < RELEVANCE_THRESHOLD or not docs:
        print("🤥 CRAG: Low retrieval confidence - falling back to web search")
        webResults = webSearch.invoke(query)
        context = "\n\n".join([r["content"] for r in webResults])
        return context, "web"
    
    else:
        print("👍 CRAG: Retrieval confidence acceptable - using local docs")
        return format_chunks(docs), "local"
    

GENERATOR_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the user's question using ONLY the context provided.
If the context does not contain eough information to answer, say exactly:
'I cannot find sufficient information in the provided context.' 
Be specific - include numbners, percentages, and fiscal year references where available.

"""

REVIEWER_SYSTEM_PROMPT = """You are a strict factual reviewer for a financial RAG system.
You will receive a question, the source context, and a generated answer.

Your job is to check:
1. Does the answer contain any claims NOT supported by the context? (hallucination)
2. Does the answer actually address the question asked?
3. Are numbers, percentages, and figures accurate relative to the context?

Respond in EXACTLY this format:
Verdict: <PASS or FAIL>
Reason: <one sentence explaining your verdict>

PASS meaans the answer is faithful to the context and addresses the question,
FAIL means the answer contains unsupported claims, wrong figuress, or avoids the question.


"""

ROUTER_SYSTEM_PROMPT = """You are a query complexity classifier for a financial RAG system.

Classify the user's question as one of:
- SIMPLE: a single factual lookup requiring one retrieval and one answer
(e.g. "What was Apple's net income in FY2024?")
- COMPLEX: requires multiple steps, comparisons, calculations, or chaining
(e.g. "Compare iPhone revenue across FY2023 and FY2024 and calculate the growth rate")
- UNKNOWN: cannot be answered from a financial document at all 
(e.g. "What is the weather in Cupertino today?")

Respond in EXACTLY this format:
Classification: <SIMPLE, COMPLEX, or UNKNOWN>
Reason: <one sentence>
"""

def generate_answer(question: str, context: str, critique: str = "") -> str:
    critiqueBlock = ""
    if critique:
        critiqueBlock = f"\n\nPrevious answer was rejected for this reason: {critique}\nPlease rewrite addressing this critique."

   # response = client.chat.completions.create(
      #  model = "openai/gpt-oss-120b",
        messages = [
            #{"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}"
                f"{critiqueBlock}"
            )}
        ]#,

       # temperature= 0,
   # )

    return tracked_llm_call(messages= messages, system= GENERATOR_SYSTEM_PROMPT)


def review_answer(question: str, context: str, answer: str) -> tuple[str, str]:
    """Returns (verdict, reason) where verdict is a PASS or FAIL."""
    
    messages = [
        #{"role": "system", "content": REVIEWER_SYSTEM_PROMPT},
        {"role": "user", "content":(
            f"Question: {question}\n\n"
            f"Source Context:\n{context}\n\n"
            f"Generated Answer:\n{answer}"

        )}
    ]

    raw = tracked_llm_call(messages= messages, system= REVIEWER_SYSTEM_PROMPT)
    verdict_match = re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match =re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f"😮‍💨 Reviewer verdict: {verdict} - {reason}")
    return verdict, reason

# NAIVE RAG PATH 

NAIVE_RAG_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the question uning ONLY THE context provided.
Be specific - include exact figures, percentages, and fiscal year references.
If the context does not contain the answer, say so directly.

"""

def run_naive_rag(question: str) -> dict:
    print("\n⚡ Path: NAIVE RAG")
    started_at = datetime.now().isoformat()

    # single retrieval
    docs, score = retrieve_with_confidence(question)
    context = "\n\n --- \n\n".join([doc.node.get_content() for doc in docs])

    # Single generation - no reviewer, no rewrite
    answer = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }],
        system = NAIVE_RAG_SYSTEM_PROMPT
    )

    print(f"Answer: {answer}")
    return {
        "path": "naive_rag",
        "question": question,
        "answer": answer,
        "retrieval_score": score,
        "started_at": started_at,
        "finished_at": datetime.now().isoformat()
    }


# THE FULL SELF CORRECTING CRAG PIPELINE

def run_crag_pipeline(question: str, max_rewrites: int = 2) ->dict:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"\n{'='*60}")

    startedAt = datetime.now().isoformat()

    # STEP 1: CRAG retrieval with confidence gate
    context, source = corrective_retrieve(question)

    # STEP 2: Generate Initial Answer
    print("\n 🦾 Generating initial answer...")
    answer = generate_answer(question, context= context)
    print(f"Answer: {answer}\n")

    # SECONDARY GATE to catch false-positive vector scores
    if answer and "I cannot find sufficient information" in answer and source == "local":
        print("🤥 CRAG: Local docs failed to answer despite high vector score. Forcing web fallback...")
        webResults = webSearch.invoke(question)
        context = "\n\n".join([r["content"] for r in webResults])
        source = "web"

        print("👍Generating answer from web context...")
        answer = generate_answer(question, context= context)
        print(f"Web Fallback Answer: {answer}\n")

    # STEP 3: Reviewer Loop
    attempts = 0
    verdict = 'FAIL'
    critique = ""
    history = []

    while verdict == 'FAIL' and attempts < max_rewrites:
        verdict, critique = review_answer(question= question, context= context, answer= answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

        if verdict == 'FAIL':
            attempts += 1
            if attempts < max_rewrites:
                print(f"\n🔁 Rewriting (attempt {attempts})...")
                answer = generate_answer(question, context, critique)
                print(f"Rewritten Answer: {answer}\n")
            else:
                print("🤥 Max rewrites reached - returning best attempt with warning")

    # final evrdict check if we exited the loop with PASS 
    if verdict != 'FAIL':
        verdict, critique = review_answer(question, context, answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

    result = {
        "question": question,
        "retrieval_source": source,
        "final_answer": answer,
        "fianl_verdict": verdict,
        "rewrite_attempts": attempts,
        "review_history": history,
        "started_at":startedAt,
        "finished_at": datetime.now().isoformat()
    }


    print(f"\n{'='*60}")
    print(f"👍 Final Answer ({verdict} after {attempts} rewrite(s)):")
    print(answer)
    print(f"{'='*60}\n")

    filename = f"traces/day11_crag_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, 'w') as f:
        json.dump(result, f, indent= 2)
    print(f"📀 Saved to {filename}")

    return result


def classify_query(question: str) -> tuple[str, str]:
    raw = tracked_llm_call(
        messages = [{"role": "user", "content": f"Question: {question}"}],
        system= ROUTER_SYSTEM_PROMPT
    )
    classification_match = re.search(r"Classification:\s*(SIMPLE|COMPLEX|UNKNOWN)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    classification = classification_match.group(1) if classification_match else "COMPLEX"
    reason = reason_match.group(1).strip() if reason_match else "Could not Parse reason."

    print(f"🦾 Router: {classification} - {reason}")
    return classification, reason

# MULTI-PATH ROUTER

def run_router(question: str) -> dict:
    global usage_tracker
    usage_tracker = TokenUsage() # reset for each question

    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    classification, reason = classify_query(question)

    if classification== 'SIMPLE': 
        result = run_naive_rag(question= question)

    elif classification == "COMPLEX":
        print("\n🤖 Path: AGENTIC RAG (CRAG + Reviewer)")
        result = run_crag_pipeline(question= question)
        result['path'] = "agentic_rag"
    
    else: # UNKNOWN
        print("\n🤥 Path: UNKNOWN - cannot answer from financial documents")
        result = {
            "path": "unknown",
            "question": question,
            "answer": "This question cannot be answered using the Apple 10-K document.",
            "started_at": datetime.now().isoformat(),
            "finished_at": datetime.now().isoformat()
        }
    
    usage_tracker.report()

    result["Classification"] = classification
    result["classification_reason"] = reason
    result["token_usage"] = {
        "prompt_tokens": usage_tracker.prompt_tokens,
        "completion_tokens": usage_tracker.completion_tokens,
        "total_calls": usage_tracker.total_calls,
        "estimated_cost_usd": usage_tracker.cost_estimate()
    }

    filename = f"traces/day12_{classification.lower()}_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, "w") as f:
        json.dump(result, f, indent = 2)
    print(f"\n📀 Saved to {filename}")

    return result



c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rodne\AppData\Local\Temp\ipykernel_28572\2373246763.py:30: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  webSearch = TavilySearchResults(max_results = 3)
Unable to verify collection 'month_1_rag_collection_v3' access: SSL handshake failed: ac-2ma9ajs-shard-00-02.clmmfwh.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1016) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0

🤖🛩️ Vector Store connection established ⚡


In [3]:
# importing necessary libraries for today

import uuid
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# improving reused RAG state from day 16

# THE STATE SCHEMA

class RAGState(TypedDict):
    # Input
    question: str

    # Router output
    classification: Optional[str]
    classification_reason: Optional[str]

    # Retrieval output
    context: Optional[str]
    retrieval_source: Optional[str]
    retrieval_score: Optional[float]

    # Generation output
    answer: Optional[str]

    # Reviewer output
    reviewer_verdict: Optional[str]
    reviewer_reason: Optional[str]

    # Rewrite tracking
    rewrite_count: int
    max_rewrites: int

    # Final flags
    warning: Optional[str]
    finished_at: Optional[str]

    # Sufficiency and reformulation tracking
    sufficiency_verdict: Optional[str]
    sufficiency_reason: Optional[str]
    retrieval_attempts: int
    max_retrieval_attempts: int
    reformulated_query: Optional[str]

    # NEW today - human approval tracking
    human_approved: Optional[bool]
    approval_reason: Optional[str]
    action_taken: Optional[str]






# THE FIVE NODES

# -----Node 1: Classify -------
def classify_node(state: RAGState) -> dict:
    print(f"\n[NODE: classify] Question: {state['question'][:60]}...")
    raw = tracked_llm_call(
        messages=[{"role": "user", "content": f"Question: {state['question']}"}],
        system= ROUTER_SYSTEM_PROMPT
    )

    classification_match = re.search(r"Classification:\s*(SIMPLE|COMPLEX|UNKNOWN)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    classification = classification_match.group(1) if classification_match else "COMPLEX"
    reason = reason_match.group(1).strip() if reason_match else "Could not parse."

    print(f" -> Classification: {classification} - {reason}")
    return {"classification": classification, "classification_reason": reason}

# ------ Node 2: Retrieve ------ # LEAB=VING THIS ONE OUT AS WE ARE GOING TO WRITE A MODIFIED VERSION OF IT IN THE NEXT CELL


# ---Node 3: Generate ----
def generate_node(state: RAGState) -> dict:
    print(f"\n[NODE: generate]")

    if state["classification"] == "UNKNOWN":
        answer = (
            "This question cannot be answered using the Apple FY2024 10-K document"
            "or available tools."
        )
        print(f" -> UNKNOWN path - returning abstention")
        return {"answer": answer}
    
    critique_block = ""
    if state.get("reviewer_reason") and state.get("rewrite_count", 0) > 0:
        critique_block = (
            f"\n\nPrevious answer was rejected: {state['reviewer_reason']}. "
            f"Please rewrite addressing this critique."
        )

    answer = tracked_llm_call(
        messages = [{
            "role":"user",
            "content": (
                f"Context:\n{state['context']}\n\n"
                f"Question: {state['question']}"
                f"{critique_block}"
            )

        }],
        system= GENERATOR_SYSTEM_PROMPT
    )
    print(f" -> Answer generated ({len(answer)} chars)")
    return {"answer": answer}


# -----Node 4: Review ----
def review_node(state: RAGState) -> dict:
    print(f"\n[NODE: review]")

    if state["classification"] == "UNKNOWN":
        print(" -> UNKNOWN path - skipping review")
        return {"reviewer_verdict": "PASS", "reviewer_reason": "Abstention accepted."}
    
    raw = tracked_llm_call(
        messages= [{
            "role": "user",
            "content": (
                f"Question: {state['question']}\n\n"
                f"Source Context:\n{state['context']}\n\n"
                f"Generated Answer:\n{state['answer']}"
            )
        }],
        system = REVIEWER_SYSTEM_PROMPT
    )

    verdict_match =re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f" -> Reviewer: {verdict} - {reason}")
    return {"reviewer_verdict" : verdict, "reviewer_reason": reason}


# ---- Node 5: Rewrite ----
def rewrite_node(state: RAGState) -> dict:
    new_count = state.get("rewrite_count", 0) + 1
    print(f"\n[NODE: rewrite] Attempt {new_count}")
    return {
        "rewrite_count" : new_count,
        "answer": None # cleared so generate_node produces a fresh answer
    }


# CONDITIONAL EDGE LOGIC

def route_after_review(state: RAGState) -> Literal["rewrite_node", "__end__"]:
    """
    Called after review_node .
    Decides whether to loop back for a rewrite or proceed to END.

    """

    verdict = state.get("reviewer_verdict", "FAIL")
    rewrite_count = state.get("rewrite_count", 0)
    max_rewrites = state.get("max_rewrites", 2)

    if verdict == "PASS":
        print(" -> Edge: PASS -> END")
        return "__end__"
    
    if rewrite_count >= max_rewrites:
        print(f" -> Edge: max rewrites ({max_rewrites}) reached -> END with warning")
        return "__end__"
    
    print(f" -> Edge: FAIL -> rewrite_node (attempt {rewrite_count + 1})")
    return "rewrite_node"


    

# MODIFIED RETRIEVE NODE FROM DAY 14
# the only difference here is it checks for a reformulated query before falling back to the original question

def retrieve_node(state: RAGState) -> dict:
    print(f"\n[NODE: retrieve]")

    if state["classification"] == "UNKNOWN":
        print(" -> UNKNOWN query - skipping retrieval")
        return{
            "context": "No context retrieved - query classified as UNKNOWN.",
            "retrieval_source": "none",
            "retrieval_score": 0.0

        }
    # Use reformulated query if one exists, otherwise the original question
    search_query = state.get("reformulated_query") or state["question"]
    print(f" -> Searching with: {search_query[: 80]}")
    
    docs, score = retrieve_with_confidence(search_query)

    if score < RELEVANCE_THRESHOLD or not docs:
        print(" -> CRAG: Low confidence - falling back to web search")
        web_results = webSearch.invoke(search_query)
        context = "\n\n".join([r["content"] for r in web_results])
        return {"context": context, "retrieval_source": "web", "retrieval_score": score}
    
    context = format_chunks(docs= docs)
    print(f" -> Local retrieval accepted (score: {score:.3f})")
    return {"context": context, "retrieval_source": "local", "retrieval_score": score}

# THE SUFFICIENCY CHECKER NODE

SUFFICIENCY_SYSTEM_PROMPT = """You are a retrieval sufficiency checker for a financial RAG system.

You will receive a question and a block of retrieved context.
Your job is to determine ONLY whether the context contains enough information
to answer the question - do not answer the question itself.

Respond in exactly this format:
Sufficiency: <SUFFICIENT or INSUFFICIENT>
Reason: <one sentence explaining why>

SUFFICIENT means the context directly contains the facts needed to answer.
INSUFFICIENT means the context is missing key facts, is off-topic, or only partially covers the question.
"""

def check_sufficiency_node(state: RAGState) -> dict:
    print(f"\n[NODE: check_sufficiency]")

    if state["classification"] == "UNKNOWN" or state.get("retrieval_source") == "web":
        # web fallback already hapened - let CRAG's existing logic handle it downstream
        print(" -> Skipping sufficiency check (UNKNOWN or already on web fallback)")
        return {"sufficiency_verdict": "SUFFICIENT", "sufficiency_reason": "Bypassed."}
    
    raw = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": (
                f"Question: {state['question']}\n\n"
                f"Retrieved Context:\n{state['context']}"
            )
        }],
        system= SUFFICIENCY_SYSTEM_PROMPT


    )

    verdict_match = re.search(r"Sufficiency:\s*(SUFFICIENT|INSUFFICIENT)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "INSUFFICIENT"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f" -> Sufficiency: {verdict} - {reason}")
    return {"sufficiency_verdict": verdict, "sufficiency_reason": reason}

# QUERY REFORMULATION NODE
REFORMULATE_SYSTEM_PROMPT = """You are a search query optimiser for a financial RAG system.

The previous search query did not retrieve sufficient context to answer the question.
Rewrite the search query to improve retrieval - use broader terms, different financial
terminology, or break the question into a more specific sub-question.

Respond with ONLY the new search query, nothing else. No explanation, no preamble.

"""

def reformulate_node(state: RAGState) -> dict:
    new_attempts = state.get("retrieval_attempts", 0) + 1
    print(f"\n[NODE: reformulate] Generating new query (will become attempt {new_attempts + 1})")

    raw = tracked_llm_call(
        messages= [{
            "role": "user",
            "content": (
                f"Original question: {state['question']}\n\n"
                f"Previous search query that failed: "
                f"{state.get('reformulated_query') or state['question']}\n\n"
                f"Why it failed: {state.get('sufficiency_reason', 'Context was insufficient.')}"

            )
        }],
        system= REFORMULATE_SYSTEM_PROMPT
    )

    new_query = raw.strip()
    print(f" -> New query: {new_query}")

    return {
        "reformulated_query": new_query,
        "retrieval_attempts": new_attempts
    }

# CONDITIONAL EDGE AFTER SUFFICIENCY CHECK

def route_after_sufficiency(state: RAGState) -> Literal["generate_node", "reformulate_node"]:
    """
    Decides whether to proceed to generation or loop back for a better search.

    """
    verdict = state.get("sufficiency_verdict", "SUFFICIENT")
    attempts = state.get("retrieval_attempts", 0)
    max_attempts = state.get("max_retrieval_attempts", 2)

    if verdict == "SUFFICIENT":
        print(" -> Edge: SUFFCIENT -> generate_node")
        return "generate_node"
    
    if attempts >= max_attempts:
        print(f" -> Edge: max retrieval attempts ({max_attempts}) reached -> generate_node anyway")
        return "generate_node"
    
    print(f" -> Edge: INSUFFICIENT -> reformulate_node (attempt {attempts + 1})")
    return "reformulate_node"

DB_PATH = "checkpoints/rag_checkpoints.db"


In [4]:
# --- Node: Human Approval ---
# This node is where the graph PAUSES via interrupt_before.
# The actual input() call here is a fallback —
# interrupt_before fires before this node runs,
# so in normal flow this node only executes after you resume.
def human_approval_node(state: RAGState) -> dict:
    print(f"\n{'='*60}")
    print(f"⏸️  HUMAN APPROVAL REQUIRED")
    print(f"{'='*60}")
    print(f"\nQuestion    : {state['question']}")
    print(f"Answer      : {state['answer']}")
    print(f"Reviewer    : {state['reviewer_verdict']} — {state['reviewer_reason']}")
    print(f"Retrieval   : {state['retrieval_source']} (score: {state['retrieval_score']:.3f})")
    print(f"Rewrites    : {state['rewrite_count']}")
    print(f"\nToken cost so far: ${usage_tracker.cost_estimate():.6f}")
    print(f"\nThis answer is about to be sent as a report.")
    print(f"Type APPROVE to continue or REJECT to cancel.\n")

    # Checking if Phase 2 already injected the decision via update_state!
    if state.get("human_approved") is not None:
        approved = state["human_approved"]
        reason = state.get("approval_reason", "Decision injected via state update")
        print(f"⚡ Decision automatically supplied via Phase 2 resume: {'APPROVED' if approved else 'REJECTED'}")
        return {"human_approved": approved, "approval_reason": reason}

    # Fallback to manual console input only if running interactively without state update
    print(f"Type APPROVE to continue or REJECT to cancel.\n")
    decision = input("Your decision: ").strip().upper()

    if decision == "APPROVE":
        print("✅ Approved — proceeding to send report")
        return {"human_approved": True, "approval_reason": "Human approved via console input"}
    else:
        print("❌ Rejected — cancelling report")
        return {
            "human_approved": False,
            "approval_reason": f"Human rejected with input: '{decision}'"
        }

# --- Node: Send Report (the expensive/irreversible action) ---
def send_report_node(state: RAGState) -> dict:
    print(f"\n[NODE: send_report]")
    print(f"📧 Sending report...")
    print(f"   To      : analyst-team@company.com (mock)")
    print(f"   Subject : Apple FY2024 10-K Analysis")
    print(f"   Body    : {state['answer'][:200]}...")
    print(f"\n✅ Report sent successfully (mock)")
    return {"action_taken": "report_sent"}


# --- Node: Rejected ---
def rejected_node(state: RAGState) -> dict:
    print(f"\n[NODE: rejected]")
    print(f"🚫 Action cancelled — {state.get('approval_reason', 'No reason given')}")
    print(f"   Answer was NOT sent")
    return {"action_taken": "rejected"}

In [5]:
def route_after_approval(
    state: RAGState
) -> Literal["send_report_node", "rejected_node"]:
    if state.get("human_approved"):
        print("  → Edge: APPROVED → send_report_node")
        return "send_report_node"
    print("  → Edge: REJECTED → rejected_node")
    return "rejected_node"

In [6]:
def build_hitl_graph(db_path: str = DB_PATH):
    conn = sqlite3.connect(db_path, check_same_thread=False)
    checkpointer = SqliteSaver(conn)

    graph = StateGraph(RAGState)

    # All nodes from previous days
    graph.add_node("classify_node", classify_node)
    graph.add_node("retrieve_node", retrieve_node)
    graph.add_node("check_sufficiency_node", check_sufficiency_node)
    graph.add_node("reformulate_node", reformulate_node)
    graph.add_node("generate_node", generate_node)
    graph.add_node("review_node", review_node)
    graph.add_node("rewrite_node", rewrite_node)

    # New nodes today
    graph.add_node("human_approval_node", human_approval_node)
    graph.add_node("send_report_node", send_report_node)
    graph.add_node("rejected_node", rejected_node)

    # Entry point
    graph.set_entry_point("classify_node")

    # Fixed edges — same as Day 16
    graph.add_edge("classify_node", "retrieve_node")
    graph.add_edge("retrieve_node", "check_sufficiency_node")
    graph.add_edge("reformulate_node", "retrieve_node")
    graph.add_edge("rewrite_node", "generate_node")
    graph.add_edge("generate_node", "review_node")

    # After review: rewrite loop or proceed to human approval
    graph.add_conditional_edges(
        "review_node",
        route_after_review,
        {
            "rewrite_node": "rewrite_node",
            # When reviewer is satisfied, go to human approval
            # instead of END — override route_after_review's "__end__"
            "__end__": "human_approval_node"
        }
    )

    # Sufficiency loop — same as Day 15
    graph.add_conditional_edges(
        "check_sufficiency_node",
        route_after_sufficiency,
        {
            "generate_node": "generate_node",
            "reformulate_node": "reformulate_node"
        }
    )

    # After human approval: send or reject
    graph.add_conditional_edges(
        "human_approval_node",
        route_after_approval,
        {
            "send_report_node": "send_report_node",
            "rejected_node": "rejected_node"
        }
    )

    # Both terminal action nodes go to END
    graph.add_edge("send_report_node", END)
    graph.add_edge("rejected_node", END)

    # KEY: interrupt_before pauses the graph BEFORE human_approval_node runs
    # State is saved to checkpoint — the node only executes after you resume
    return graph.compile(
        checkpointer=checkpointer,
        interrupt_before=["human_approval_node"]
    )


hitl_graph = build_hitl_graph()
print("✅ Human-in-the-loop graph compiled")

try:
    print(hitl_graph.get_graph().draw_ascii())
except Exception:
    print("ASCII visualisation unavailable")

✅ Human-in-the-loop graph compiled
ASCII visualisation unavailable


In [7]:
def run_hitl_phase1(question: str) -> tuple[RAGState, str]:
    """
    Phase 1: Run the graph until it hits the interrupt before human_approval_node.
    Returns the paused state and thread_id for Phase 2.
    """
    global usage_tracker
    usage_tracker = TokenUsage()

    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    print(f"\n{'='*60}")
    print(f"PHASE 1: Running graph until approval gate")
    print(f"Thread ID: {thread_id}")
    print(f"{'='*60}")

    initial_state: RAGState = {
        "question": question,
        "classification": None,
        "classification_reason": None,
        "context": None,
        "retrieval_source": None,
        "retrieval_score": None,
        "answer": None,
        "reviewer_verdict": None,
        "reviewer_reason": None,
        "rewrite_count": 0,
        "max_rewrites": 2,
        "warning": None,
        "finished_at": None,
        "sufficiency_verdict": None,
        "sufficiency_reason": None,
        "retrieval_attempts": 0,
        "max_retrieval_attempts": 2,
        "reformulated_query": None,
        "human_approved": None,
        "approval_reason": None,
        "action_taken": None,
    }

    # Graph runs, hits interrupt_before human_approval_node, and stops
    paused_state = hitl_graph.invoke(initial_state, config=config)

    print(f"\n⏸️  Graph paused — awaiting human approval")
    print(f"Answer preview: {str(paused_state.get('answer', ''))[:200]}")
    print(f"Reviewer verdict: {paused_state.get('reviewer_verdict')}")
    print(f"Thread ID for resume: {thread_id}")

    return paused_state, thread_id


def run_hitl_phase2(thread_id: str, approve: bool = True) -> RAGState:
    """
    Phase 2: Resume the graph after human decision.
    Injects the human_approved flag into the paused state and resumes.
    """
    config = {"configurable": {"thread_id": thread_id}}

    decision = "APPROVE" if approve else "REJECT"
    print(f"\n{'='*60}")
    print(f"PHASE 2: Resuming with decision: {decision}")
    print(f"Thread ID: {thread_id}")
    print(f"{'='*60}\n")

    # Update the state with the human decision before resuming
    hitl_graph.update_state(
        config,
        {
            "human_approved": approve,
            "approval_reason": f"Human {'approved' if approve else 'rejected'} via console"
        }
    )

    # Resume from checkpoint — graph continues from human_approval_node
    final_state = hitl_graph.invoke(None, config=config)
    final_state["finished_at"] = datetime.now().isoformat()

    print(f"\n{'='*60}")
    print(f"✅ Graph completed")
    print(f"Action taken: {final_state.get('action_taken')}")
    print(f"{'='*60}\n")

    filename = (
        f"traces/day17_hitl_{thread_id[:8]}_"
        f"{'approved' if approve else 'rejected'}_"
        f"{datetime.now().strftime('%H%M%S')}.json"
    )
    with open(filename, "w") as f:
        json.dump(dict(final_state), f, indent=2)
    print(f"📁 Saved to {filename}")

    return final_state

In [9]:
# ── Scenario 1: Approve ───────────────────────────────────────────────────
print("\n" + "#"*60)
print("SCENARIO 1: Run → Pause → APPROVE → Send report")
print("#"*60)

paused_1, tid_1 = run_hitl_phase1(
    "What was Apple's total revenue for fiscal year 2024 "
    "and how does it break down by product segment?"
)
# Graph pauses here — inspect the answer before deciding
final_1 = run_hitl_phase2(tid_1, approve=True)
print(f"Action taken: {final_1['action_taken']}")


# ── Scenario 2: Reject ────────────────────────────────────────────────────
print("\n" + "#"*60)
print("SCENARIO 2: Run → Pause → REJECT → Cancel")
print("#"*60)

paused_2, tid_2 = run_hitl_phase1(
    "What were Apple's major risk factors disclosed "
    "in the FY2024 10-K related to competition?"
)
# Reject this one — confirm the report is NOT sent
final_2 = run_hitl_phase2(tid_2, approve=False)
print(f"Action taken: {final_2['action_taken']}")
assert final_2["action_taken"] == "rejected", "❌ Safety failure: report sent despite rejection"
print("✅ Safety confirmed: report was NOT sent after rejection")


# ── Scenario 3: Inspect checkpoint mid-pause ─────────────────────────────
print("\n" + "#"*60)
print("SCENARIO 3: Pause → Inspect → Then decide")
print("#"*60)

paused_3, tid_3 = run_hitl_phase1(
    "How did Apple's gross margin change between FY2023 and FY2024 "
    "and what does management attribute this to?"
)

# Inspect before deciding
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute(
    "SELECT COUNT(*) FROM checkpoints WHERE thread_id = ?", (tid_3,)
)
count = cursor.fetchone()[0]
conn.close()
print(f"\n🔍 Checkpoint inspection: {count} checkpoints saved for this thread")
print(f"State is fully preserved — safe to decide now or later")

# Approve after inspection
final_3 = run_hitl_phase2(tid_3, approve=True)


############################################################
SCENARIO 1: Run → Pause → APPROVE → Send report
############################################################

PHASE 1: Running graph until approval gate
Thread ID: e3c85e05-6f97-4610-b724-66e0766445da

[NODE: classify] Question: What was Apple's total revenue for fiscal year 2024 and how ...
 -> Classification: SIMPLE - The question seeks specific factual figures (total revenue and segment breakdown) that can be answered with a single retrieval from Apple's FY2024 financial report.

[NODE: retrieve]
 -> Searching with: What was Apple's total revenue for fiscal year 2024 and how does it break down b
📣 Top retrieval score : 0.860 (threshold: 0.5)
 -> Local retrieval accepted (score: 0.860)

[NODE: check_sufficiency]
 -> Sufficiency: SUFFICIENT - The context provides Apple’s total 2024 net sales ($391,035 million) and the detailed product‑segment sales figures needed for the breakdown.
 -> Edge: SUFFCIENT -> generate_node

[NOD